# GAN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

def get_real_samples(n=1000):
    x = torch.empty(n, 1).uniform_(-3.0, 3.0)
    y = -x
    return torch.cat([x, y], dim=1)

real_data = get_real_samples(1000)
print(real_data.shape)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=1):
        super().__init__()
        self.value_layer = nn.Linear(latent_dim, 1)
        self.weight_layer = nn.Linear(latent_dim, 1)

    def forward(self, z):
        value = self.value_layer(z)
        weight = self.weight_layer(z)
        weighted_value = weight * value
        return torch.cat([weighted_value, -weighted_value], dim=1)

class Discriminator(nn.Module):
    def __init__(self, input_dim=2):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

In [ ]:
G = Generator(latent_dim=1)
D = Discriminator(input_dim=2)

opt_g = torch.optim.Adam(G.parameters(), lr=5e-4)
opt_d = torch.optim.Adam(D.parameters(), lr=5e-4)
criterion = nn.BCELoss(reduction='sum')

real_labels = torch.ones(64, 1)
fake_labels = torch.zeros(64, 1)

for epoch in range(1, 3001):
    idx = torch.randint(0, 1000, (64,))
    real_batch = real_data[idx]

    z = torch.randn(64, 1)
    fake_batch = G(z)

    d_loss_real = criterion(D(real_batch), real_labels)
    d_loss_fake = criterion(D(fake_batch.detach()), fake_labels)
    d_loss = d_loss_real + d_loss_fake

    opt_d.zero_grad()
    d_loss.backward()
    opt_d.step()

    z = torch.randn(64, 1)
    fake_batch = G(z)
    g_loss = criterion(D(fake_batch), real_labels)

    opt_g.zero_grad()
    g_loss.backward()
    opt_g.step()

    if epoch % 300 == 0:
        print(f"Epoch {epoch:4d} | D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f}")

print("Training complete.")

In [ ]:
noise = torch.randn(1000, 1)
generated = G(noise).detach()

real = get_real_samples(1000)

plt.figure(figsize=(6,6))
plt.scatter(real[:,0], real[:,1], label="Real (y=-x)", alpha=0.5)
plt.scatter(generated[:,0], generated[:,1], label="Generated", alpha=0.5)
plt.legend()
plt.title("GAN Learning y = -x Distribution")
plt.show()

# Diffusion Model

In [ ]:
T = 100
beta = torch.linspace(0.0001, 0.02, T)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

def add_noise(x0, t):
    noise = torch.randn_like(x0)
    a = alpha_bar[t].view(-1, 1)
    x_t = torch.sqrt(a) * x0 + torch.sqrt(1 - a) * noise
    return x_t, noise

In [ ]:
class DiffModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x, t):
        t = t.view(-1, 1) / T
        return self.net(torch.cat([x, t], dim=1))

In [ ]:
model = DiffModel()
opt = torch.optim.Adam(model.parameters(), lr=0.002)
loss_fn = nn.MSELoss()

for epoch in range(2000):
    x0 = get_real_samples(128)
    t = torch.randint(0, T, (128,))
    x_t, noise = add_noise(x0, t)
    
    pred_noise = model(x_t, t)
    loss = loss_fn(pred_noise, noise)
    
    opt.zero_grad()
    loss.backward()
    opt.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    x = torch.randn(1000, 2)
    for t in range(T-1, -1, -1):
        t_tensor = torch.full((1000,), t)
        pred_noise = model(x, t_tensor)
        
        a = alpha[t]
        a_bar = alpha_bar[t]
        b = beta[t]
        
        z = torch.randn_like(x) if t > 0 else 0
        x = (1 / torch.sqrt(a)) * (x - ((1 - a) / torch.sqrt(1 - a_bar)) * pred_noise) + torch.sqrt(b) * z

gen_data = x + torch.randn(1000, 2) * 0.08
real_data = get_real_samples(1000)

plt.figure(figsize=(6,6))
plt.scatter(real_data[:,0], real_data[:,1], label="Real", alpha=0.5)
plt.scatter(gen_data[:,0], gen_data[:,1], label="Generated", alpha=0.5)
plt.legend()
plt.title("Diffusion Learning y = -x Distribution")
plt.show()